# UK Dataset — Stratified Sampling (Parquet-Optimized Version)

**Objective:** Sample **1B tokens** from 60.7B tokens while ensuring that every CC-MAIN batch directory is covered

**Data format (confirmed):**
- Each `CC-MAIN-YYYY-WW/` contains `chunk_*.parquet` files
- Fields: `text, id, dump, url, date, file_path, language, language_score, token_count, index`
- **`token_count` is precomputed**, so tiktoken is skipped and the speed is significantly improved

| Item | Value |
|---|---|
| Total token count | 60,714,120,289 |
| Total number of files | 27,168 |
| Target token count | 1,000,000,000 |
| Base sampling ratio | ~1.647% |

In [1]:
#!pip install pyarrow tqdm ipywidgets

In [3]:
!pip install tqdm

In [4]:
# ── Cell 1: Imports & configuration ─────────────────────────────────────────
import os, json, random, time
from pathlib import Path
from collections import defaultdict
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

# ════════════════════════════════════════════════════════════════
DATA_ROOT  = Path('/home/jovyan/data/UK')
OUTPUT_DIR = Path('./uk_sample')
# ════════════════════════════════════════════════════════════════

OUTPUT_FILE = OUTPUT_DIR / 'uk_1B_sample.parquet'
STATS_FILE  = OUTPUT_DIR / 'sampling_stats.json'

TARGET_TOKENS = 1_000_000_000
TOTAL_TOKENS  = 60_714_120_289
SAMPLE_RATIO  = TARGET_TOKENS / TOTAL_TOKENS   # ≈ 1.647%
RANDOM_SEED   = 42

random.seed(RANDOM_SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'DATA_ROOT    : {DATA_ROOT}')
print(f'OUTPUT_FILE  : {OUTPUT_FILE}')
print(f'SAMPLE_RATIO : {SAMPLE_RATIO:.4%}')

DATA_ROOT    : /home/jovyan/data/UK
OUTPUT_FILE  : uk_sample/uk_1B_sample.parquet
SAMPLE_RATIO : 1.6471%


In [5]:
# ── Cell 2: Scan all CC-MAIN directories and find .parquet files ──────────
dir_files = {}   # { 'CC-MAIN-2013-20': [Path, Path, ...] }

for batch_dir in sorted(DATA_ROOT.iterdir()):
    if not batch_dir.is_dir() or not batch_dir.name.startswith('CC-MAIN'):
        continue
    files = sorted(batch_dir.glob('*.parquet'))
    if files:
        dir_files[batch_dir.name] = files

all_dirs = sorted(dir_files.keys())
n_dirs   = len(all_dirs)
total_files_scanned = sum(len(v) for v in dir_files.values())

print(f'Number of CC-MAIN directories found : {n_dirs}')
print(f'Parquet files found : {total_files_scanned}')
print(f'\nNumber of files per directory (first 10 entries):')
for d in all_dirs[:10]:
    print(f'  {d}: {len(dir_files[d])} files')
if n_dirs > 10:
    print(f'  ... another {n_dirs-10} directories')

Number of CC-MAIN directories found : 109
Parquet files found : 1040

Number of files per directory (first 10 entries):
  CC-MAIN-2013-20: 7 files
  CC-MAIN-2013-48: 6 files
  CC-MAIN-2014-10: 6 files
  CC-MAIN-2014-15: 6 files
  CC-MAIN-2014-23: 7 files
  CC-MAIN-2014-35: 7 files
  CC-MAIN-2014-41: 7 files
  CC-MAIN-2014-42: 6 files
  CC-MAIN-2014-49: 5 files
  CC-MAIN-2014-52: 6 files
  ... another 99 directories


In [6]:
# ── Cell 3: Use progress.json to obtain the actual token count for each directory ─────
# More accurate than using file-count proportions: allocate quotas directly by token share

dir_tokens = {}
for d in all_dirs:
    progress = DATA_ROOT / d / 'progress.json'
    if progress.exists():
        with open(progress) as f:
            info = json.load(f)
        dir_tokens[d] = info.get('total_tokens', 0)
    else:
        dir_tokens[d] = 0   # For directories without progress.json, estimate later using the number of files

sum_known_tokens = sum(dir_tokens.values())
print(f'Total tokens collected from progress.json: {sum_known_tokens:,}')
print(f'Total tokens declared in metadata          : {TOTAL_TOKENS:,}')
print(f'\nTop 5 directories by token count:')
for d, t in sorted(dir_tokens.items(), key=lambda x: -x[1])[:5]:
    print(f'  {d}: {t:>14,} tokens')

Total tokens collected from progress.json: 735,319,053,119
Total tokens declared in metadata          : 60,714,120,289

Top 5 directories by token count:
  CC-MAIN-2023-50: 13,353,032,483 tokens
  CC-MAIN-2022-21: 11,839,454,849 tokens
  CC-MAIN-2021-31: 10,851,979,676 tokens
  CC-MAIN-2021-43: 10,829,918,998 tokens
  CC-MAIN-2022-27: 10,400,432,812 tokens


In [7]:
# ── Cell 4: Stratified quota calculation (by token share, at least 1 file per directory)─
# Target tokens per directory = SAMPLE_RATIO × the directory's total tokens
# Number of sampled files = ceil(number of files in the directory × SAMPLE_RATIO), at least 1

import math

quota_per_dir = {}
for d in all_dirs:
    n_files = len(dir_files[d])
    # Sample by file proportion, at least 1 and not exceeding the actual number of files
    n_to_pick = max(1, min(math.ceil(n_files * SAMPLE_RATIO), n_files))
    quota_per_dir[d] = n_to_pick

total_selected_files = sum(quota_per_dir.values())
print(f'Actual number of selected files  : {total_selected_files}')
print(f'\nDirectory quotas (first 10 entries):')
for d in all_dirs[:10]:
    print(f'  {d}: {len(dir_files[d])} files → sample {quota_per_dir[d]}')

Actual number of selected files  : 109

Directory quotas (first 10 entries):
  CC-MAIN-2013-20: 7 files → sample 1
  CC-MAIN-2013-48: 6 files → sample 1
  CC-MAIN-2014-10: 6 files → sample 1
  CC-MAIN-2014-15: 6 files → sample 1
  CC-MAIN-2014-23: 7 files → sample 1
  CC-MAIN-2014-35: 7 files → sample 1
  CC-MAIN-2014-41: 7 files → sample 1
  CC-MAIN-2014-42: 6 files → sample 1
  CC-MAIN-2014-49: 5 files → sample 1
  CC-MAIN-2014-52: 6 files → sample 1


In [8]:
# ── Cell 5: Randomly select specific files ───────────────────────────────────
selected_files = []
for d in all_dirs:
    pool   = dir_files[d]
    quota  = quota_per_dir[d]
    chosen = random.sample(pool, quota)
    selected_files.extend(chosen)

random.shuffle(selected_files)

print(f'Final number of selected files : {len(selected_files)}')
print(f'\nSample (first 5 files):')
for f in selected_files[:5]:
    print(f'  {f.parent.name}/{f.name}')

Final number of selected files : 109

Sample (first 5 files):
  CC-MAIN-2017-47/chunk_3.parquet
  CC-MAIN-2017-09/chunk_6.parquet
  CC-MAIN-2019-39/chunk_3.parquet
  CC-MAIN-2019-04/chunk_5.parquet
  CC-MAIN-2016-22/chunk_0.parquet


In [9]:
# ── Cell 6: Main sampling loop (streaming writes, low memory)──────────────────────
import pyarrow as pa
from collections import defaultdict
from tqdm import tqdm

tokens_per_dir_quota = TARGET_TOKENS // n_dirs
print(f"Target contribution per directory: {tokens_per_dir_quota:,} tokens")

files_by_dir = defaultdict(list)
for fp in selected_files:
    files_by_dir[fp.parent.name].append(fp)

total_tokens_written = 0
total_rows_written   = 0
files_done           = 0
t0                   = time.time()
stats_per_dir        = defaultdict(lambda: {'files': 0, 'tokens': 0, 'rows': 0})

pbar = tqdm(total=TARGET_TOKENS, unit='tok', unit_scale=True, desc='Sampling progress', mininterval=0.5)

writer = None   # Streaming ParquetWriter

for dir_name in all_dirs:
    dir_quota = tokens_per_dir_quota
    dir_collected = 0

    for fp in files_by_dir[dir_name]:
        if dir_collected >= dir_quota:
            break
        try:
            table = pq.read_table(fp)
        except Exception as e:
            print(f'[WARN] Skip {fp.name}: {e}')
            continue

        token_counts = table.column('token_count').to_numpy()
        cumsum = token_counts.cumsum()
        need = dir_quota - dir_collected

        if cumsum[-1] <= need:
            take_n = len(token_counts)
            take_tokens = int(cumsum[-1])
        else:
            idx = int((cumsum >= need).argmax()) + 1
            take_n = idx
            take_tokens = int(cumsum[idx-1])
            table = table.slice(0, take_n)

        if writer is None:
            writer = pq.ParquetWriter(OUTPUT_FILE, table.schema, compression='snappy')

        writer.write_table(table)
        del table

        dir_collected        += take_tokens
        total_tokens_written += take_tokens
        total_rows_written   += take_n
        stats_per_dir[dir_name]['files']  += 1
        stats_per_dir[dir_name]['tokens'] += take_tokens
        stats_per_dir[dir_name]['rows']   += take_n
        files_done += 1
        pbar.update(take_tokens)

        if files_done % 5 == 0:
            elapsed = time.time() - t0
            tps = total_tokens_written / max(elapsed, 1e-6)
            print(f"  [{files_done}] {dir_name}: cumulative {total_tokens_written/1e6:.0f}M tokens, covered {len(stats_per_dir)}/{n_dirs} directories, speed {tps/1e6:.1f}M tok/s")

if writer is not None:
    writer.close()
pbar.close()

elapsed_total = time.time() - t0
print(f'\n{"="*55}')
print(f'✅ Sampling complete！')
print(f'   Total tokens     : {total_tokens_written:,}')
print(f'   Total rows        : {total_rows_written:,}')
print(f'   Processed files    : {files_done}')
print(f'   Covered directories    : {len(stats_per_dir)} / {n_dirs}')
print(f'   Total elapsed time        : {elapsed_total/60:.1f} minutes')
print(f'   Average speed      : {total_tokens_written/elapsed_total/1e6:.1f}M tok/s')
print(f'   Output files      : {OUTPUT_FILE}')
print(f'   Output size      : {OUTPUT_FILE.stat().st_size/1024/1024:.1f} MB')

Target contribution per directory: 9,174,311 tokens


Sampling progress:   5%|▍         | 45.9M/1.00G [01:16<26:30, 600ktok/s]

  [5] CC-MAIN-2014-23: cumulative 46M tokens, covered 5/109 directories, speed 0.6M tok/s


Sampling progress:   9%|▉         | 91.8M/1.00G [02:40<26:30, 571ktok/s]

  [10] CC-MAIN-2014-52: cumulative 92M tokens, covered 10/109 directories, speed 0.6M tok/s


...

  [100] CC-MAIN-2024-38: cumulative 918M tokens, covered 100/109 directories, speed 0.6M tok/s


Sampling progress:  96%|█████████▋| 963M/1.00G [26:56<01:06, 553ktok/s]

  [105] CC-MAIN-2025-08: cumulative 963M tokens, covered 105/109 directories, speed 0.6M tok/s


Sampling progress: 1.00Gtok [27:59, 596ktok/s]                         


✅ Sampling complete！
   Total tokens     : 1,000,144,505
   Total rows        : 1,614,593
   Processed files    : 109
   Covered directories    : 109 / 109
   Total elapsed time        : 28.0 minutes
   Average speed      : 0.6M tok/s
   Output files      : uk_sample/uk_1B_sample.parquet
   Output size      : 2803.1 MB


In [10]:
# ── Cell 7: Save statistics report ────────────────────────────────────────
stats = {
    'dataset': 'UK',
    'sampling_method': 'stratified_by_cc_main_batch_min1_per_dir',
    'random_seed': RANDOM_SEED,
    'target_tokens': TARGET_TOKENS,
    'total_tokens_written': total_tokens_written,
    'total_rows_written': total_rows_written,
    'files_processed': files_done,
    'dirs_covered': len(stats_per_dir),
    'total_dirs': n_dirs,
    'elapsed_minutes': round(elapsed_total / 60, 2),
    'output_size_mb': round(OUTPUT_FILE.stat().st_size / 1024 / 1024, 1),
    'per_directory': {k: v for k, v in sorted(stats_per_dir.items())}
}

with open(STATS_FILE, 'w') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f'Statistics report saved: {STATS_FILE}\n')
print(f'{"directories":<25} {"files":>5} {"Tokens":>14} {"Rows":>10}')
print('-' * 60)
for d, s in sorted(stats_per_dir.items()):
    print(f'{d:<25} {s["files"]:>5} {s["tokens"]:>14,} {s["rows"]:>10,}')
print('-' * 60)
print(f'{"Total":<25} {files_done:>5} {total_tokens_written:>14,} {total_rows_written:>10,}')

Statistics report saved: uk_sample/sampling_stats.json

directories                           files         Tokens         Rows
------------------------------------------------------------
CC-MAIN-2013-20               1      9,174,542     14,813
CC-MAIN-2013-48               1      9,176,021     13,965
CC-MAIN-2014-10               1      9,175,292     14,068
CC-MAIN-2014-15               1      9,174,764     13,511
CC-MAIN-2014-23               1      9,174,556     13,253
CC-MAIN-2014-35               1      9,177,942     12,728
CC-MAIN-2014-41               1      9,179,139     13,409
CC-MAIN-2014-42               1      9,174,325     13,880
CC-MAIN-2014-49               1      9,176,819     13,309
CC-MAIN-2014-52               1      9,174,519     14,027
CC-MAIN-2015-06               1      9,174,733     13,349
CC-MAIN-2015-11               1      9,174,343     14,084
CC-MAIN-2015-14               1      9,176,053     13,276
CC-MAIN-2015-18               1      9,174,441     13,410

In [11]:
# ── Cell 8: Validate output file ────────────────────────────────────────
verify = pq.read_table(OUTPUT_FILE)
print(f'Total rows in output file : {verify.num_rows:,}')
print(f'Output file column names   : {verify.column_names}')
print(f'token_count sum: {sum(verify.column("token_count").to_numpy()):,}')

print('\nPreview of the first 3 records:')
for i in range(3):
    row = verify.slice(i, 1).to_pylist()[0]
    print(f'\n── Record {i+1} ──')
    print(f'  dump  : {row["dump"]}')
    print(f'  url   : {row["url"]}')
    print(f'  tokens: {row["token_count"]}')
    print(f'  text  : {row["text"][:150]}...')

Total rows in output file : 1,614,593
Output file column names   : ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count', 'index']
token_count sum: 1,000,144,505

Preview of the first 3 records:

── Record 1 ──
  dump  : CC-MAIN-2013-20
  url   : http://www.nextgenerationrecords.co.uk/
  tokens: 578
  text  : Brisk & Fracus / Darwin
A. Shine On Me
AA. Nothing In Our Way
AAA. Believe Again
A. I Found You
AA. I Found You (Nu Foundation Remix)
AAA. Believe Aga...

── Record 2 ──
  dump  : CC-MAIN-2013-20
  url   : http://www.nhm.ac.uk/nature-online/art-nature-imaging/collections/macgillivray/detail.dsml?ID=194&listPageURL=browsecat.dsml%3FSubject%3DBirds%26sort%3DTitle
  tokens: 104
  text  : Common Kestrel - Falco tinnunculus
Common name: Common Kestrel
Scientific name: Falco tinnunculus
Accession number: 194
Materials: Watercolour on pape...

── Record 3 ──
  dump  : CC-MAIN-2013-20
  url   : http://www.no1magazine.co.uk/articles/jacksonkids.html

## Speed estimate

| Stage | Estimate |
|---|---|
| Cell 2-5 scan + quota calculation + file selection | < 5 seconds |
| **Cell 6 main sampling** | **1-3 minutes** |
| Cell 7-8 output statistics + validation | < 5 seconds |
| **Total** | **≈ 1.5-3.5 minutes** |

**Why it is fast:**
- Parquet uses columnar storage, so the notebook can directly read and accumulate the `token_count` column, **completely skipping tiktoken**
- Each file is read in one pass (pyarrow is implemented in C++)
- stop immediately after reaching the target, possibly processing only around ~100 files

## Output files
- `uk_1B_sample.parquet` — sample (Snappy compressed, ~3-5 GB)
- `sampling_stats.json` — sampling statistics for each CC-MAIN batch

## Description for the paper
> *We employ stratified random sampling across Common Crawl batches (CC-MAIN-YYYY-WW), allocating at least one file per batch to ensure temporal coverage. The number of files sampled per batch is proportional to the batch's file count. Token counts are taken directly from the pre-computed `token_count` field. Random seed = 42 for reproducibility.*